<a href="https://colab.research.google.com/github/yuyunk-minilab/pengenalan-pola/blob/main/pelatihanmodelGCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
sst2023_kdd_cup_2015_path = kagglehub.dataset_download('sst2023/kdd-cup-2015')

print('Data source import complete.')


# Pipeline lengkap di Kaggle: sampling 10.000 enrollment, fitur entropi, dan GNN

**Sebelum mulai:** tambahkan dataset KDD Cup 2015 lewat **+ Add Input** di panel kanan, dan aktifkan GPU di **Settings → Accelerator** (untuk Sel 5).

Jalankan Sel 1–5 berurutan:
1. Mencari file dataset di `/kaggle/input` dan mengatur parameter
2. Langkah 1–3: sampling, penggabungan, pembersihan
3. Langkah 4–7: penyaringan, penanda waktu, fitur dasar dan entropi
4. Membangun graf transisi per enrollment
5. Melatih GCN dan membandingkannya dengan regresi logistik

Semua hasil tersimpan di `/kaggle/working`. Klik **Save Version** lalu unduh dari tab **Output**.


In [ ]:
# ===== SEL 1: CARI DATA DI KAGGLE DAN ATUR PARAMETER =====

import glob, os
import numpy as np
import pandas as pd

N_SAMPLE = 10000                 # jumlah enrollment yang diambil
SEED = 42                        # ganti (mis. 7) untuk uji ketahanan
INPUT_DIR = "/kaggle/input"      # lokasi dataset (hanya-baca)
base = "/kaggle/working"         # lokasi simpan semua hasil
os.makedirs(base, exist_ok=True)

def cari(*nama):
    for n in nama:
        h = sorted(glob.glob(f"{INPUT_DIR}/**/{n}", recursive=True))
        if h:
            return h[0]
    return None

f = {"log": cari("log_train.csv", "train.csv", "train.xlsx"),
     "enroll": cari("enrollment_train.csv"),
     "object": cari("object.csv"),
     "truth": cari("truth_train.csv"),
     "date": cari("date.csv")}
for k, v in f.items():
    print(f"{k:7}: {v or 'TIDAK DITEMUKAN'}")
if not (f["log"] and f["enroll"] and f["object"]):
    print("\nIsi /kaggle/input:")
    for p in sorted(glob.glob(f"{INPUT_DIR}/**/*", recursive=True))[:50]:
        print("  ", p)
    raise RuntimeError("File wajib (log_train, enrollment_train, object) tidak ditemukan. Tambahkan dataset lewat '+ Add Input'.")


log    : /kaggle/input/datasets/sst2023/kdd-cup-2015/train/train/log_train.csv
enroll : /kaggle/input/datasets/sst2023/kdd-cup-2015/train/train/enrollment_train.csv
object : /kaggle/input/datasets/sst2023/kdd-cup-2015/object.csv
truth  : /kaggle/input/datasets/sst2023/kdd-cup-2015/train/train/truth_train.csv
date   : /kaggle/input/datasets/sst2023/kdd-cup-2015/date.csv


In [ ]:
# ===== SEL 2: LANGKAH 1-3 DENGAN SAMPLING 10.000 ENROLLMENT (VERSI TAHAN ERROR) =====

for v in ("f", "N_SAMPLE", "SEED", "base"):
    if v not in globals():
        raise RuntimeError(f"Variabel '{v}' belum ada. Jalankan Sel 1 dulu, baru Sel 2.")

rep = {}
ALIAS = {"enroll_id": "enrollment_id", "enrollmentid": "enrollment_id", "username": "username",
         "user_id": "username", "course": "course_id", "courseid": "course_id", "label": "dropout",
         "truth": "dropout", "timestamp": "time", "datetime": "time", "action": "event"}

def rapikan(df):
    """Samakan nama kolom (huruf kecil, tanpa spasi, alias umum)."""
    df.columns = [ALIAS.get(c.strip().lower(), c.strip().lower()) for c in df.columns]
    return df

# ---------- Langkah 1: periksa file dan ambil sampel enrollment ----------
enroll = rapikan(pd.read_csv(f["enroll"]))
obj = rapikan(pd.read_csv(f["object"]))
print("Kolom enrollment:", list(enroll.columns))
for kol in ("enrollment_id", "course_id"):
    if kol not in enroll.columns:
        raise RuntimeError(f"Kolom '{kol}' tidak ada di {f['enroll']}. Kolom yang ada: {list(enroll.columns)}")
enroll["enrollment_id"] = pd.to_numeric(enroll.enrollment_id, errors="coerce")
enroll = enroll.dropna(subset=["enrollment_id"])
enroll["enrollment_id"] = enroll.enrollment_id.astype("int64")
rep["enroll_rows_population"] = len(enroll)
rep["enroll_courses_population"] = enroll.course_id.nunique()

if f["truth"]:
    # deteksi otomatis: file truth bisa dengan atau tanpa baris header
    awal = pd.read_csv(f["truth"], header=None, nrows=1)
    ada_header = not str(awal.iloc[0, 0]).strip().isdigit()
    truth = pd.read_csv(f["truth"], header=0 if ada_header else None)
    truth = truth.iloc[:, :2]
    truth.columns = ["enrollment_id", "dropout"]
    truth["enrollment_id"] = pd.to_numeric(truth.enrollment_id, errors="coerce")
    truth["dropout"] = pd.to_numeric(truth.dropout, errors="coerce")
    truth = truth.dropna().astype({"enrollment_id": "int64", "dropout": "int64"})
    print(f"truth_train: {'dengan' if ada_header else 'tanpa'} header, {len(truth):,} baris")
    e = enroll.merge(truth, on="enrollment_id", how="inner")
    if e.empty:
        raise RuntimeError("Tidak ada enrollment_id yang cocok antara enrollment_train dan truth_train.")
    strata = ["course_id", "dropout"]
    rep["dropout_rate_population"] = round(e.dropout.mean(), 4)
else:
    e = enroll.copy()
    strata = ["course_id"]
    print("PERINGATAN: truth_train.csv tidak ada; sampling hanya distratifikasi per kursus.")

frac = min(1.0, N_SAMPLE / len(e))
sampel = e.groupby(strata, group_keys=False).sample(frac=frac, random_state=SEED)
ids = set(sampel.enrollment_id)
rep["enroll_sampled"] = len(ids)
if "dropout" in sampel:
    rep["dropout_rate_sample"] = round(sampel.dropout.mean(), 4)

print("Membaca log secara bertahap dan menyaring enrollment terpilih...")
if f["log"].endswith((".xlsx", ".xls")):
    raw = rapikan(pd.read_excel(f["log"]))
    rep["log_rows_read"] = len(raw)
    log = raw[pd.to_numeric(raw.enrollment_id, errors="coerce").isin(ids)]
else:
    parts, n_read = [], 0
    for c in pd.read_csv(f["log"], chunksize=1_000_000):
        c = rapikan(c)
        if "enrollment_id" not in c.columns:
            raise RuntimeError(f"Kolom enrollment_id tidak ada di log. Kolom yang ada: {list(c.columns)}")
        n_read += len(c)
        parts.append(c[pd.to_numeric(c.enrollment_id, errors="coerce").isin(ids)])
        print(f"  dibaca {n_read:,} baris", end="\r")
    log = pd.concat(parts, ignore_index=True)
    rep["log_rows_read"] = n_read
print()
kurang = {"enrollment_id", "time", "source", "event", "object"} - set(log.columns)
if kurang:
    raise RuntimeError(f"Kolom log tidak lengkap: {kurang}. Kolom yang ada: {list(log.columns)}")
log["enrollment_id"] = pd.to_numeric(log.enrollment_id, errors="coerce").astype("int64")
rep["log_rows_sample"] = len(log)
rep["log_nulls"] = int(log.isna().sum().sum())

# ---------- Langkah 2: gabungkan tabel ----------
df = log.merge(sampel, on="enrollment_id", how="left")
if f["date"]:
    dates = pd.read_csv(f["date"]).rename(columns={"from": "course_from", "to": "course_to"})
    df = df.merge(dates, on="course_id", how="left")
cat = obj.drop_duplicates("module_id").set_index("module_id")["category"]
df["object_category"] = df["object"].map(cat)
rep["object_matched_pct"] = round(df.object_category.notna().mean() * 100, 1)

# ---------- Langkah 3: bersihkan data ----------
df["time"] = pd.to_datetime(df["time"], errors="coerce")
rep["time_unparsed"] = int(df.time.isna().sum())
df = df.dropna(subset=["time"])
sebelum = len(df)
df = df.drop_duplicates(subset=["enrollment_id", "time", "source", "event", "object"])
rep["duplicates_removed"] = sebelum - len(df)
if f["date"]:
    awal = pd.to_datetime(df.course_from)
    akhir = pd.to_datetime(df.course_to) + pd.Timedelta(days=1)
    sebelum = len(df)
    df = df[(df.time >= awal) & (df.time < akhir)]
    rep["outside_course_window_removed"] = sebelum - len(df)
df = df.sort_values(["enrollment_id", "time"]).reset_index(drop=True)
rep["log_rows_clean"] = len(df)
rep["enrollments_with_log"] = df.enrollment_id.nunique()
rep["courses_clean"] = df.course_id.nunique()
rep["time_min"] = str(df.time.min())
rep["time_max"] = str(df.time.max())

sampel.to_csv(f"{base}/sample_enrollments_seed{SEED}.csv", index=False)
out_log = f"{base}/log_clean_sample_seed{SEED}.csv"
df.to_csv(out_log, index=False)
print("\n===== RINGKASAN LANGKAH 1-3 =====")
for k, v in rep.items():
    print(f"{k}: {v}")
print(f"\nTersimpan: {out_log}")


Kolom enrollment: ['enrollment_id', 'username', 'course_id']
truth_train: tanpa header, 120,542 baris
Membaca log secara bertahap dan menyaring enrollment terpilih...
  dibaca 8,157,277 baris

===== RINGKASAN LANGKAH 1-3 =====
enroll_rows_population: 120542
enroll_courses_population: 39
dropout_rate_population: 0.7929
enroll_sampled: 10000
dropout_rate_sample: 0.7929
log_rows_read: 8157277
log_rows_sample: 677448
log_nulls: 0
object_matched_pct: 54.0
time_unparsed: 0
duplicates_removed: 27433
outside_course_window_removed: 0
log_rows_clean: 650015
enrollments_with_log: 10000
courses_clean: 39
time_min: 2013-10-27 00:25:22
time_max: 2014-08-01 15:34:15

Tersimpan: /kaggle/working/log_clean_sample_seed42.csv


In [ ]:
# ===== SEL 3: LANGKAH 4-7 (FITUR DASAR DAN ENTROPI) =====
MIN_EVENTS = 10      # langkah 4: minimal event per enrollment
UTC_OFFSET = 8       # langkah 5: geser jam ke waktu Beijing
SESSION_GAP = 60     # langkah 5: menit jeda pemisah sesi
EVENTS = ["video", "problem", "access", "wiki", "discussion", "navigate", "page_close"]

def entropy_norm(counts, k):
    c = counts.to_numpy(dtype=float)
    tot = c.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        p = np.where(tot > 0, c / tot, 0)
        h = -np.nansum(np.where(p > 0, p * np.log2(p), 0), axis=1)
    return pd.Series(h / np.log2(k), index=counts.index)

def cond_entropy(d0, key):
    cols = list(dict.fromkeys(["enrollment_id", key, "session", "event"]))
    d = d0[cols].copy()
    d["prev"] = d.groupby(["enrollment_id", "session"])["event"].shift(1)
    d = d.dropna(subset=["prev"])
    if d.empty:
        return pd.Series(dtype=float)
    j = d.groupby([key, "prev", "event"]).size().rename("n").reset_index()
    j["p"] = j.n / j.groupby(key).n.transform("sum")
    j["plogp"] = -j.p * np.log2(j.p)
    h_joint = j.groupby(key).plogp.sum()
    m = j.groupby([key, "prev"]).p.sum().rename("p").reset_index()
    m["plogp"] = -m.p * np.log2(m.p)
    h_prev = m.groupby(key).plogp.sum()
    return (h_joint - h_prev) / np.log2(len(EVENTS))

d = pd.read_csv(f"{base}/log_clean_sample_seed{SEED}.csv", parse_dates=["time"])
d = d.sort_values(["enrollment_id", "time"]).reset_index(drop=True)

# ---------- Langkah 4: saring enrollment ----------
n_ev = d.groupby("enrollment_id").size()
keep = n_ev[n_ev >= MIN_EVENTS].index
print(f"Langkah 4: enrollment {n_ev.size}, dibuang (< {MIN_EVENTS} event): {n_ev.size - keep.size}, tersisa: {keep.size}")
d = d[d.enrollment_id.isin(keep)].copy()

# ---------- Langkah 5: penanda waktu ----------
d["hour"] = (d.time + pd.Timedelta(hours=UTC_OFFSET)).dt.hour
if "course_from" in d.columns:
    start = pd.to_datetime(d.course_from).dt.normalize()
else:
    start = d.groupby("course_id").time.transform("min").dt.normalize()
    print("PERINGATAN: date.csv tidak ada; hari ke- diperkirakan dari log paling awal tiap kursus")
d["day"] = (d.time.dt.normalize() - start).dt.days.clip(0, 29)
d["week"] = (d.day // 7).clip(upper=3) + 1
gap = d.groupby("enrollment_id").time.diff().dt.total_seconds().div(60)
d["session"] = (gap.isna() | (gap > SESSION_GAP)).groupby(d.enrollment_id).cumsum()

g = d.groupby("enrollment_id")
ev = pd.crosstab(d.enrollment_id, d.event).reindex(columns=EVENTS, fill_value=0)

# ---------- Langkah 6: fitur dasar ----------
feat = pd.DataFrame(index=ev.index)
feat["course_id"] = g.course_id.first()
for e_ in EVENTS:
    feat[f"n_{e_}"] = ev[e_]
feat["n_total"] = ev.sum(axis=1)
feat["active_days"] = g.day.nunique()
feat["n_sessions"] = g.session.nunique()
s = d.groupby(["enrollment_id", "session"]).time.agg(["min", "max"])
feat["mean_session_min"] = (s["max"] - s["min"]).dt.total_seconds().div(60).groupby(level=0).mean()
feat["n_unique_objects"] = g.object.nunique()
feat["share_browser"] = (d.source == "browser").groupby(d.enrollment_id).mean()

# ---------- Langkah 7: fitur entropi ----------
feat["H_act"] = entropy_norm(ev, len(EVENTS))
feat["H_hour"] = entropy_norm(pd.crosstab(d.enrollment_id, d.hour).reindex(columns=range(24), fill_value=0), 24)
feat["H_day"] = entropy_norm(pd.crosstab(d.enrollment_id, d.day).reindex(columns=range(30), fill_value=0), 30)
feat["H_trans"] = cond_entropy(d, "enrollment_id")

d["ew"] = d.enrollment_id.astype(str) + "_" + d.week.astype(str)
ev_w = pd.crosstab(d.ew, d.event).reindex(columns=EVENTS, fill_value=0)
h_act_w = entropy_norm(ev_w, len(EVENTS))
h_trans_w = cond_entropy(d, "ew")
for w in range(1, 5):
    ids_w = feat.index.astype(str) + f"_{w}"
    feat[f"H_act_w{w}"] = h_act_w.reindex(ids_w).fillna(0).to_numpy()
    feat[f"H_trans_w{w}"] = h_trans_w.reindex(ids_w).fillna(0).to_numpy()
    feat[f"active_w{w}"] = ev_w.reindex(ids_w).sum(axis=1).fillna(0).gt(0).astype(int).to_numpy()
wk = feat[[f"H_act_w{w}" for w in range(1, 5)]].to_numpy()
feat["H_act_volatility"] = wk.std(axis=1)
feat["H_act_trend"] = np.polyfit(np.arange(1, 5), wk.T, 1)[0]
feat["inactive_weeks"] = 4 - feat[[f"active_w{w}" for w in range(1, 5)]].sum(axis=1)
feat["H_trans"] = feat["H_trans"].fillna(0)
if "dropout" in d.columns:
    feat["dropout"] = g.dropout.first()

feat = feat.reset_index()
out_feat = f"{base}/features_sample_seed{SEED}.csv"
feat.to_csv(out_feat, index=False)
print(f"\nTersimpan: {out_feat} ({feat.shape[0]} enrollment, {feat.shape[1]} kolom)")
print(feat[["H_act", "H_hour", "H_day", "H_trans", "H_act_volatility"]].describe().round(3))
if "dropout" in feat:
    print("\nProporsi dropout:", round(feat.dropout.mean(), 4))


Langkah 4: enrollment 10000, dibuang (< 10 event): 3819, tersisa: 6181

Tersimpan: /kaggle/working/features_sample_seed42.csv (6181 enrollment, 35 kolom)
          H_act    H_hour     H_day   H_trans  H_act_volatility
count  6181.000  6181.000  6181.000  6181.000          6181.000
mean      0.704     0.313     0.215     0.408             0.279
std       0.121     0.225     0.209     0.106             0.088
min      -0.000    -0.000    -0.000     0.000             0.000
25%       0.641     0.137     0.000     0.339             0.257
50%       0.714     0.306     0.191     0.415             0.298
75%       0.790     0.490     0.369     0.485             0.334
max       0.980     0.901     0.904     0.690             0.473

Proporsi dropout: 0.6954


In [ ]:
# ===== SEL 4: BANGUN GRAF TRANSISI PER ENROLLMENT (GNN) =====

d = pd.read_csv(f"{base}/log_clean_sample_seed{SEED}.csv", parse_dates=["time"])
feat = pd.read_csv(f"{base}/features_sample_seed{SEED}.csv")
if "dropout" not in feat.columns:
    raise RuntimeError("Kolom dropout tidak ada: truth_train.csv tidak ditemukan di dataset Kaggle yang dipakai.")
print(f"Log: {len(d):,} baris | Enrollment berfitur: {len(feat):,} | Proporsi dropout: {feat.dropout.mean():.3f}")

# Setiap enrollment = satu graf dengan 7 node (jenis event).
# Edge = frekuensi transisi antarevent di dalam sesi (jeda <= 60 menit).
EVENTS = ["video", "problem", "access", "wiki", "discussion", "navigate", "page_close"]
K = len(EVENTS)
ev_idx = {e: i for i, e in enumerate(EVENTS)}

ids = feat.enrollment_id.to_numpy()
pos = {e: i for i, e in enumerate(ids)}
N = len(ids)

d = d[d.enrollment_id.isin(pos)].sort_values(["enrollment_id", "time"]).copy()
gap = d.groupby("enrollment_id").time.diff().dt.total_seconds().div(60)
d["session"] = (gap.isna() | (gap > 60)).groupby(d.enrollment_id).cumsum()
d["ev"] = d.event.map(ev_idx)
d = d.dropna(subset=["ev"])
d["ev"] = d.ev.astype(int)
d["prev"] = d.groupby(["enrollment_id", "session"]).ev.shift(1)

# matriks hitungan transisi (N, 7, 7)
C = np.zeros((N, K, K), dtype=np.float32)
t = d.dropna(subset=["prev"])
np.add.at(C, (t.enrollment_id.map(pos).to_numpy(), t.prev.astype(int).to_numpy(), t.ev.to_numpy()), 1)

# adjacency simetris ternormalisasi + self-loop: D^-1/2 (A + I) D^-1/2
A = C + C.transpose(0, 2, 1)
tot = A.sum(axis=(1, 2), keepdims=True)
A = np.divide(A, tot, out=np.zeros_like(A), where=tot > 0) + np.eye(K, dtype=np.float32)[None]
dinv = 1.0 / np.sqrt(A.sum(axis=2))
A_norm = (dinv[:, :, None] * A * dinv[:, None, :]).astype(np.float32)

# fitur node: proporsi event, log jumlah event (diskalakan), proporsi hari aktif untuk event tsb
cnt = pd.crosstab(d.enrollment_id, d.ev).reindex(index=ids, columns=range(K), fill_value=0).to_numpy(float)
share = cnt / np.maximum(cnt.sum(axis=1, keepdims=True), 1)
logc = np.log1p(cnt) / max(np.log1p(cnt).max(), 1e-9)
d["date"] = d.time.dt.normalize()
days = (d.groupby(["enrollment_id", "ev"]).date.nunique().unstack()
          .reindex(index=ids, columns=range(K)).fillna(0).to_numpy(float))
days = np.clip(days / 30.0, 0, 1)
X = np.stack([share, logc, days], axis=-1).astype(np.float32)   # (N, 7, 3)
y = feat.dropout.to_numpy().astype(np.float32)
print(f"Graf: {N:,} | node per graf: {K} | fitur node: {X.shape[2]} | rata-rata transisi per graf: {C.sum(axis=(1,2)).mean():.1f}")


Log: 650,015 baris | Enrollment berfitur: 6,181 | Proporsi dropout: 0.695
Graf: 6,181 | node per graf: 7 | fitur node: 3 | rata-rata transisi per graf: 98.0


In [ ]:
# ===== SEL 5: LATIH GNN (GCN) DAN BANDINGKAN DENGAN REGRESI LOGISTIK =====
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

torch.manual_seed(SEED); np.random.seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"

# pembagian data 70/15/15, terstratifikasi menurut dropout (sama untuk semua model)
all_ix = np.arange(N)
tr, te = train_test_split(all_ix, test_size=0.15, stratify=y, random_state=SEED)
tr, va = train_test_split(tr, test_size=0.15 / 0.85, stratify=y[tr], random_state=SEED)

class GCN(nn.Module):
    def __init__(self, f_in, h=32):
        super().__init__()
        self.l1, self.l2 = nn.Linear(f_in, h), nn.Linear(h, h)
        self.head = nn.Sequential(nn.Linear(2 * h, h), nn.ReLU(), nn.Dropout(0.2), nn.Linear(h, 1))
    def forward(self, A, X):
        H = torch.relu(A @ self.l1(X))
        H = torch.relu(A @ self.l2(H))
        g = torch.cat([H.mean(dim=1), H.max(dim=1).values], dim=1)   # pooling tingkat graf
        return self.head(g).squeeze(1)

At, Xt, yt = torch.tensor(A_norm), torch.tensor(X), torch.tensor(y)
model = GCN(X.shape[2]).to(dev)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
pw = torch.tensor((1 - y[tr].mean()) / y[tr].mean(), dtype=torch.float32, device=dev)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pw)

def predict(ix):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(At[ix].to(dev), Xt[ix].to(dev))).cpu().numpy()

best, bad, state = -1, 0, None
for epoch in range(200):
    model.train()
    perm = np.random.permutation(tr)
    for b in range(0, len(perm), 256):
        ix = perm[b:b + 256]
        opt.zero_grad()
        loss = loss_fn(model(At[ix].to(dev), Xt[ix].to(dev)), yt[ix].to(dev))
        loss.backward(); opt.step()
    auc_va = roc_auc_score(y[va], predict(va))
    if auc_va > best:
        best, bad = auc_va, 0
        state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    else:
        bad += 1
    if bad >= 20:
        break
model.load_state_dict(state)
print(f"GCN berhenti di epoch {epoch + 1}, AUC validasi terbaik = {best:.3f}")

def metrics(name, yt_, p):
    yhat = (p >= 0.5).astype(int)
    return {"Model": name, "AUC": roc_auc_score(yt_, p), "F1": f1_score(yt_, yhat),
            "Precision": precision_score(yt_, yhat, zero_division=0), "Recall": recall_score(yt_, yhat)}

base_cols = [f"n_{e}" for e in EVENTS] + ["n_total", "active_days", "n_sessions",
             "mean_session_min", "n_unique_objects", "share_browser"]
ent_cols = ["H_act", "H_hour", "H_day", "H_trans", "H_act_volatility", "H_act_trend", "inactive_weeks"]
rows = []
for name, cols in [("Logistic regression A: baseline", base_cols),
                   ("Logistic regression B: baseline + entropy", base_cols + ent_cols)]:
    Xf = feat[cols].fillna(0).to_numpy()
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
    lr.fit(Xf[tr], y[tr])
    rows.append(metrics(name, y[te], lr.predict_proba(Xf[te])[:, 1]))
rows.append(metrics("GCN: activity transition graph", y[te], predict(te)))

res = pd.DataFrame(rows).round(3)
out_csv = f"{base}/results_gnn_seed{SEED}.csv"
res.to_csv(out_csv, index=False)
print(f"\nHasil pada data uji (n = {len(te):,}):")
print(res.to_string(index=False))
print(f"\nTersimpan: {out_csv}  (unduh dari tab Output setelah Save Version)")


GCN berhenti di epoch 64, AUC validasi terbaik = 0.854

Hasil pada data uji (n = 928):
                                    Model   AUC    F1  Precision  Recall
          Logistic regression A: baseline 0.842 0.853      0.853   0.853
Logistic regression B: baseline + entropy 0.864 0.858      0.873   0.843
           GCN: activity transition graph 0.842 0.832      0.872   0.795

Tersimpan: /kaggle/working/results_gnn_seed42.csv  (unduh dari tab Output setelah Save Version)


In [ ]:
import shutil
from IPython.display import FileLink
shutil.make_archive("/kaggle/working/hasil_kaggle", "zip", "/kaggle/working")
FileLink("hasil_kaggle.zip")